## Сканирование параметра β в модели SIR

### Подготовка окружения

Подключаем необходимые пакеты и модули.

In [ ]:
using DrWatson
@quickactivate "project"
include(srcdir("SIRPetri.jl"))
using .SIRPetri
using DataFrames, CSV, Plots

### Параметры сканирования

Задаём диапазон значений коэффициента заражения β и фиксированные параметры.

- `β_range` — диапазон значений β от 0.1 до 0.8 с шагом 0.05
- `γ_fixed` — фиксированный коэффициент выздоровления
- `tmax` — максимальное время симуляции

In [ ]:
β_range = 0.1:0.05:0.8
γ_fixed = 0.1
tmax = 100.0

### Цикл сканирования

Для каждого значения β запускаем детерминированную симуляцию
и записываем:
- `peak_I` — максимальное число инфицированных (пик эпидемии)
- `final_R` — конечное число выздоровевших

In [ ]:
results = []
for β in β_range
    net, u0, _ = build_sir_network(β, γ_fixed)
    df = simulate_deterministic(net, u0, (0.0, tmax), saveat = 0.5, rates = [β, γ_fixed])
    peak_I = maximum(df.I)
    final_R = df.R[end]
    push!(results, (β = β, peak_I = peak_I, final_R = final_R))
end

### Сохранение результатов

Преобразуем результаты в DataFrame и сохраняем в CSV-файл.

In [ ]:
df_scan = DataFrame(results)
CSV.write(datadir("sir_scan.csv"), df_scan)

### Визуализация

Строим график зависимости пика эпидемии (Peak I) и конечного числа
выздоровевших (Final R) от коэффициента заражения β.

In [ ]:
p = plot(
    df_scan.β,
    [df_scan.peak_I df_scan.final_R],
    label = ["Peak I" "Final R"],
    marker = :circle,
    xlabel = "β (infection rate)",
    ylabel = "Population",
)
savefig(plotsdir("sir_scan.png"))

### Завершение

In [ ]:
println("Сканирование β завершено. Результат в data/sir_scan.csv")